# Grover's Search Algorithm

Grover's algorithm finds a marked item in an unsorted database of N items
in O(sqrt(N)) queries. Here we search 3 qubits (N=8).

In [ ]:
import cirq
import numpy as np

## Oracle and Diffusion

- Oracle marks target with phase flip (multi-controlled Z)
- Diffusion reflects about the mean (2|s><s| - I)

In [ ]:
def grover_oracle(qubits, target):
    n = len(qubits)
    target_bits = format(target, f"0{n}b")
    ops = []
    for i, bit in enumerate(reversed(target_bits)):
        if bit == "0":
            ops.append(cirq.X(qubits[i]))
    if n == 2:
        ops.append(cirq.CZ(qubits[0], qubits[1]))
    else:
        ops.append(cirq.CCX(qubits[0], qubits[1], qubits[2]))
        ops.append(cirq.CZ(qubits[2], qubits[1]))
        ops.append(cirq.CCX(qubits[0], qubits[1], qubits[2]))
    for i, bit in enumerate(reversed(target_bits)):
        if bit == "0":
            ops.append(cirq.X(qubits[i]))
    return ops

def diffusion(qubits):
    n = len(qubits)
    ops = []
    ops.extend(cirq.H(q) for q in qubits)
    ops.extend(cirq.X(q) for q in qubits)
    if n == 2:
        ops.append(cirq.CZ(qubits[0], qubits[1]))
    else:
        ops.append(cirq.CCX(qubits[0], qubits[1], qubits[2]))
        ops.append(cirq.CZ(qubits[2], qubits[1]))
        ops.append(cirq.CCX(qubits[0], qubits[1], qubits[2]))
    ops.extend(cirq.X(q) for q in qubits)
    ops.extend(cirq.H(q) for q in qubits)
    return ops

## Grover Search (N=8, target=|101\u27e9)

Optimal iterations: floor(pi/4 * sqrt(N)) = 2

In [ ]:
n = 3
N = 2**n
qubits = cirq.LineQubit.range(n)
target = 5
sim = cirq.Simulator()

num_iters = int(np.pi / 4 * np.sqrt(N))
print(f"N={N}, target=|{target:03b}\u27e9, iterations={num_iters}\n")

circuit = cirq.Circuit(
    [cirq.H(q) for q in qubits],
    num_iters * (grover_oracle(qubits, target) + diffusion(qubits)),
)
print(circuit)

sv = sim.simulate(circuit).final_state_vector
probs = np.abs(sv) ** 2
print("\nProbabilities:")
for i in range(N):
    marker = " <--" if i == target else ""
    print(f"  |{i:03b}\u27e9: {probs[i]:.4f}{marker}")

## Iteration Count Sweep

In [ ]:
for iters in range(6):
    sweep = cirq.Circuit(
        [cirq.H(q) for q in qubits],
        iters * (grover_oracle(qubits, target) + diffusion(qubits)),
    )
    sv = sim.simulate(sweep).final_state_vector
    p = abs(sv[target]) ** 2
    print(f"  {iters} iterations: P(target) = {p:.4f}")

## Measurement Statistics

In [ ]:
meas = cirq.Circuit(
    [cirq.H(q) for q in qubits],
    num_iters * (grover_oracle(qubits, target) + diffusion(qubits)),
    cirq.measure(*qubits, key="m"),
)
result = sim.run(meas, repetitions=1000)
counts = result.histogram(key="m")
for k in sorted(counts):
    print(f"  |{k:03b}\u27e9: {counts[k]} ({counts[k]/10:.1f}%)")